# PAUL Open Model - DPO V2 Training (Kaggle)

This notebook executes the DPO V2 corrective experiment on Kaggle.
It strictly validates the private SFT adapter and captures full provenance before launching training.

In [ ]:
!git clone https://github.com/paul-foundry/paul-open.git /kaggle/working/paul-open
%cd /kaggle/working/paul-open
!pip install -r requirements.txt

## 1. Provenance and Validation
Compute hashes and gather environment details.

In [ ]:
import os
import sys
import json
import hashlib
import subprocess
import torch
import transformers
import trl
import peft
import bitsandbytes
import platform
import yaml

def hash_file(path):
    h = hashlib.sha256()
    if not os.path.exists(path):
        return None
    with open(path, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def get_git_commit():
    try:
        return subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode('utf-8').strip()
    except:
        return 'unknown'

ADAPTER_DIR = "/kaggle/input/paul-open-sft-adapter"

print("--- PROVENANCE AUDIT ---")
git_commit = get_git_commit()
print(f"Git Commit: {git_commit}")

# Validate adapter
if not os.path.isdir(ADAPTER_DIR):
    print(f"CRITICAL ERROR: Adapter directory {ADAPTER_DIR} not found.")
    sys.exit(1)

adapter_config_path = os.path.join(ADAPTER_DIR, "adapter_config.json")
safetensors_path = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")

config_hash = hash_file(adapter_config_path)
safetensors_hash = hash_file(safetensors_path)

print(f"Adapter config SHA-256: {config_hash}")
print(f"Adapter model SHA-256: {safetensors_hash}")

if not config_hash or not safetensors_hash:
    print("CRITICAL ERROR: Adapter artifacts missing.")
    sys.exit(1)

with open(adapter_config_path, 'r') as f:
    adapter_config = json.load(f)

print(f"Adapter Base Model: {adapter_config.get('base_model_name_or_path')}")

data_hash = hash_file("data/train/dpo_v2_corrective.jsonl")
dpo_config_hash = hash_file("configs/training/dpo.yaml")
train_script_hash = hash_file("scripts/train.py")

print(f"Dataset SHA-256: {data_hash}")
print(f"Training config SHA-256: {dpo_config_hash}")
print(f"Train script SHA-256: {train_script_hash}")

print("\n--- ENVIRONMENT ---")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"TRL: {trl.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"BitsAndBytes: {bitsandbytes.__version__}")
print(f"CUDA: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CRITICAL ERROR: No GPU found!")
    sys.exit(1)

manifest = {
    "git_commit": git_commit,
    "adapter_hashes": {
        "config": config_hash,
        "safetensors": safetensors_hash
    },
    "adapter_base_model": adapter_config.get('base_model_name_or_path'),
    "dataset_hash": data_hash,
    "config_hash": dpo_config_hash,
    "environment": {
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
    }
}
with open('dpo_v2_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print("\nManifest saved to dpo_v2_manifest.json.")


## 2. Dry Run
Ensure everything parses and loads correctly without starting the heavy training loop.

In [ ]:
!python scripts/train.py --model configs/models/gemma4_12b_it.yaml \
                         --training configs/training/dpo.yaml \
                         --data data/train/dpo_v2_corrective.jsonl \
                         --adapter /kaggle/input/paul-open-sft-adapter \
                         --dry-run

## 3. Execute Training
If dry run succeeded, launch the actual DPO training.

In [ ]:
!python scripts/train.py --model configs/models/gemma4_12b_it.yaml \
                         --training configs/training/dpo.yaml \
                         --data data/train/dpo_v2_corrective.jsonl \
                         --adapter /kaggle/input/paul-open-sft-adapter